# Cross-Species Generalization Test & Inference Speed Benchmark

This notebook evaluates human-promoter-trained models on **mouse** and **plant** promoter data
(cross-species generalization) and benchmarks **inference speed** for teacher vs student.

**Models:** Teacher (DNABERT-2, 117.1M) vs Hybrid-CNN-3Layer Student (26.3M)

**Generalization datasets:**
- Mouse promoters (same kingdom — Mammalia) from DeePromoter/CNNPromoterData
- Plant promoters (different kingdom — Plantae, 13 species) from zhangtaolab HuggingFace

**Setup:** Add teacher + student notebook outputs as Kaggle inputs. GPU T4 x 2, Internet ON.


In [ ]:
!pip install transformers datasets accelerate scikit-learn einops

# Configuration & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import gc
import os
import json
import time
import sys
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertConfig,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# ===== Configurable Constants =====
TEACHER_NOTEBOOK_SLUG = "ml-project-kaggle15916a635e"
TEACHER_CHECKPOINT_PATH = f"/kaggle/input/{TEACHER_NOTEBOOK_SLUG}/lr_2e-05_results/checkpoint-4375"  # Best: 96.71%
TEACHER_FALLBACK_PATH = f"/kaggle/input/{TEACHER_NOTEBOOK_SLUG}/human_promoter_model"                # Fallback: 96.40%
MODEL_NAME = "quietflamingo/dnabert2-no-flashattention"
MAX_LENGTH = 300
BATCH_SIZE = 16
NUM_EPOCHS = 10
SEED = 42

# ===== Hyperparameter Grid =====
ALPHAS = [0.3, 0.5, 0.7]
TEMPERATURES = [1.0, 3.0, 5.0]

# ===== Reproducibility =====
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===== GPU Info =====
device = "cuda" if torch.cuda.is_available() else "cpu"
num_gpus = torch.cuda.device_count()
print(f"Device: {device}")
print(f"Number of GPUs: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

print(f"\nHyperparameter grid: {len(ALPHAS)} alphas x {len(TEMPERATURES)} temps = {len(ALPHAS)*len(TEMPERATURES)} combos per architecture")
print(f"Alphas: {ALPHAS}")
print(f"Temperatures: {TEMPERATURES}")

# Load Dataset & Tokenize

In [ ]:
# Load HUMAN promoter dataset from InstaDeepAI
print("Loading human promoter dataset from InstaDeepAI...")
raw_dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")

# Filter for promoter_all task only
def filter_promoter_all(example):
    return example["task"] == "promoter_all"

print("Filtering for promoter_all task...")
train_data_raw = raw_dataset["train"].filter(filter_promoter_all)
test_data_raw = raw_dataset["test"].filter(filter_promoter_all)

# Prepare data
def prepare_data(example):
    return {
        "sequence": example["sequence"].upper(),
        "label": int(example["label"]),
    }

train_data = train_data_raw.map(prepare_data)
test_data = test_data_raw.map(prepare_data)

print(f"Training samples: {len(train_data)}, Test samples: {len(test_data)}")

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def tokenize_func(examples):
    return tokenizer(examples["sequence"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print(f"Tokenizing with max_length={MAX_LENGTH}...")
tokenized_train = train_data.map(tokenize_func, batched=True)
tokenized_test = test_data.map(tokenize_func, batched=True)

# Set format for PyTorch and rename label -> labels for HuggingFace Trainer
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

train_dataset = tokenized_train
eval_dataset = tokenized_test

print(f"Tokenization complete!")
print(f"Train columns: {train_dataset.column_names}")
print(f"Sample input_ids shape: {train_dataset[0]['input_ids'].shape}")

# Load Teacher Model

In [ ]:
# Auto-discover teacher model path under /kaggle/input/
# Kaggle mounts notebook outputs at varying depths, e.g.:
#   /kaggle/input/notebooks/username/slug/lr_2e-05_results/checkpoint-4375/
import glob
from safetensors.torch import load_file as load_safetensors
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

def find_teacher_model():
    """Search all Kaggle inputs (recursively) for teacher checkpoint files."""
    base = "/kaggle/input"
    
    # Priority 1: Best checkpoint (96.71%)
    for filename in ["model.safetensors", "pytorch_model.bin"]:
        matches = glob.glob(f"{base}/**/lr_2e-05_results/checkpoint-4375/{filename}", recursive=True)
        if matches:
            return os.path.dirname(matches[0]), "best checkpoint (96.71%)"
    
    # Priority 2: Phase 1 model (96.40%)
    for filename in ["model.safetensors", "pytorch_model.bin"]:
        matches = glob.glob(f"{base}/**/human_promoter_model/{filename}", recursive=True)
        if matches:
            return os.path.dirname(matches[0]), "Phase 1 model (96.40%)"
    
    # Priority 3: Any checkpoint with config.json
    matches = glob.glob(f"{base}/**/human_promoter_results/checkpoint-*/config.json", recursive=True)
    if matches:
        return os.path.dirname(matches[0]), "fallback checkpoint"
    
    return None, None

teacher_path, teacher_desc = find_teacher_model()

if teacher_path is None:
    print("Contents of /kaggle/input/:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth < 3:
            indent = "  " * depth
            print(f"{indent}{os.path.basename(root)}/")
    raise FileNotFoundError(
        "Teacher model not found!\n"
        "Please add the teacher notebook output as a Kaggle input:\n"
        "  File -> Add Data -> Your Datasets -> Notebook Output"
    )

print(f"Found teacher: {teacher_desc}")
print(f"Path: {teacher_path}")

# Load config from hub and patch missing fields
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.num_labels = 2
config.pad_token_id = tokenizer.pad_token_id  # DNABERT-2 bert_layers.py:39 needs this

# Get DNABERT-2's custom model class from its auto_map
# (bypasses from_pretrained's meta-device context that breaks ALiBi tensor math)
class_ref = config.auto_map["AutoModelForSequenceClassification"]
model_class = get_class_from_dynamic_module(class_ref, MODEL_NAME)

# Direct instantiation on CPU — no meta device wrapping
teacher_model = model_class(config)

# Load fine-tuned checkpoint weights
weights_file = os.path.join(teacher_path, "model.safetensors")
if os.path.exists(weights_file):
    state_dict = load_safetensors(weights_file)
else:
    weights_file = os.path.join(teacher_path, "pytorch_model.bin")
    state_dict = torch.load(weights_file, map_location="cpu", weights_only=True)

teacher_model.load_state_dict(state_dict)
print(f"Loaded fine-tuned weights from: {os.path.basename(weights_file)}")

teacher_model.eval()

teacher_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Teacher model loaded successfully!")
print(f"Teacher parameters: {teacher_params:,} ({teacher_params/1e6:.1f}M)")

# Student Architecture Definition


In [ ]:
from transformers.modeling_outputs import SequenceClassifierOutput

# ===== HYBRID CNN-TRANSFORMER STUDENT =====
class DNAHybridStudent(nn.Module):
    def __init__(self, layers=2):
        super().__init__()
        config = BertConfig.from_pretrained("quietflamingo/dnabert2-no-flashattention", num_hidden_layers=layers)
        self.bert = BertForSequenceClassification(config).bert
        self.conv = nn.Conv1d(in_channels=768, out_channels=128, kernel_size=9, padding=4)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.classifier = nn.Linear(128, 2)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        x = sequence_output.permute(0, 2, 1)
        x = torch.relu(self.conv(x))
        x = self.pool(x).squeeze(-1)
        logits = self.classifier(x)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)


# ===== PARAMETER COUNTING =====
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


print("DNAHybridStudent and count_parameters defined.")

# Load Best Hybrid Student Checkpoint


In [ ]:
# ===== LOAD BEST HYBRID STUDENT MODEL =====
import glob

# --- Find model.pt (Hybrid) ---
# Search broadly for any model.pt in Kaggle inputs
all_model_pts = glob.glob("/kaggle/input/**/*.pt", recursive=True)
print("All .pt files found in /kaggle/input/:")
for p in all_model_pts:
    print(f"  {p}")

# Also list the input directory structure for debugging
print("\n/kaggle/input/ top-level contents:")
for item in os.listdir("/kaggle/input/"):
    print(f"  {item}/")
    subpath = os.path.join("/kaggle/input", item)
    if os.path.isdir(subpath):
        for sub in os.listdir(subpath)[:10]:
            print(f"    {sub}")

# Search for model.pt specifically (Hybrid saves as model.pt)
hybrid_paths = glob.glob("/kaggle/input/**/model.pt", recursive=True)
print(f"\nFound model.pt files: {hybrid_paths}")

hybrid_student = DNAHybridStudent(layers=3)
if hybrid_paths:
    state_dict = torch.load(hybrid_paths[0], map_location=device, weights_only=True)
    hybrid_student.load_state_dict(state_dict)
    print(f"Loaded Hybrid-CNN-3Layer from: {hybrid_paths[0]}")
else:
    print("WARNING: No model.pt found! Using random weights.")
hybrid_student.eval()
hybrid_student.to(device)
print(f"Hybrid params: {count_parameters(hybrid_student):,}")

# --- Ensure teacher is on GPU ---
teacher_model.to(device)
teacher_model.eval()
print(f"Teacher moved to {device}")

# --- Quick verification on human test set ---
print(f"\n{'='*60}")
print("QUICK VERIFICATION ON HUMAN TEST SET")
print(f"{'='*60}")

from torch.utils.data import DataLoader

eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

def quick_eval(model, loader, model_name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    print(f"  {model_name:<25} Accuracy: {acc:.2%}  F1: {f1:.4f}")
    return acc, f1

teacher_human_acc, teacher_human_f1 = quick_eval(teacher_model, eval_loader, "Teacher (DNABERT-2)")
hybrid_human_acc, hybrid_human_f1 = quick_eval(hybrid_student, eval_loader, "Hybrid-CNN-3Layer")

print("Verification complete!")

# Cross-Species Generalization: Mouse & Plant Promoters


In [ ]:
# ===== DOWNLOAD MOUSE PROMOTER DATA =====
# Source: DeePromoter (Oubounyt et al., 2019) — EPDnew mouse promoters, 300bp

import urllib.request

MOUSE_FILES = {
    "Mouse_tata.fa": "https://raw.githubusercontent.com/solovictor/CNNPromoterData/master/Mouse_tata.fa",
    "Mouse_non_tata.fa": "https://raw.githubusercontent.com/solovictor/CNNPromoterData/master/Mouse_non_tata.fa",
    "Mouse_nonprom.fa": "https://raw.githubusercontent.com/solovictor/CNNPromoterData/master/Mouse_nonprom.fa",
}

mouse_dir = "/kaggle/working/mouse_data"
os.makedirs(mouse_dir, exist_ok=True)

for fname, url in MOUSE_FILES.items():
    fpath = os.path.join(mouse_dir, fname)
    if not os.path.exists(fpath):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, fpath)
    else:
        print(f"Already have {fname}")

# ===== LOAD PLANT MULTI-SPECIES PROMOTER DATA =====
# Source: zhangtaolab/plant-multi-species-core-promoters (HuggingFace)
# 13 plant species, 300bp, binary classification (active promoter vs inactive)
print("\nLoading plant multi-species promoter dataset from HuggingFace...")
plant_dataset_raw = load_dataset("zhangtaolab/plant-multi-species-core-promoters")
plant_test = plant_dataset_raw["test"]
print(f"Plant test set: {len(plant_test)} samples")

# Show species distribution
from collections import Counter
species_counts = Counter(plant_test["sp"])
print("Plant species in test set:")
for sp, cnt in species_counts.most_common():
    print(f"  {sp}: {cnt}")

# ===== PARSE FASTA FILES =====
def parse_fasta(filepath):
    sequences = []
    current_seq = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current_seq:
                    sequences.append("".join(current_seq).upper())
                    current_seq = []
            else:
                current_seq.append(line)
        if current_seq:
            sequences.append("".join(current_seq).upper())
    return sequences

mouse_tata = parse_fasta(os.path.join(mouse_dir, "Mouse_tata.fa"))
mouse_nontata = parse_fasta(os.path.join(mouse_dir, "Mouse_non_tata.fa"))
mouse_nonprom = parse_fasta(os.path.join(mouse_dir, "Mouse_nonprom.fa"))

print(f"\nMouse TATA promoters: {len(mouse_tata)}")
print(f"Mouse non-TATA promoters: {len(mouse_nontata)}")
print(f"Mouse non-promoters: {len(mouse_nonprom)}")

# Build mouse dataset: promoter (1) vs non-promoter (0)
mouse_promoters = mouse_tata + mouse_nontata
mouse_sequences = mouse_promoters + mouse_nonprom
mouse_labels = [1] * len(mouse_promoters) + [0] * len(mouse_nonprom)

print(f"Total mouse: {len(mouse_sequences)} (promoters: {len(mouse_promoters)}, non-promoters: {len(mouse_nonprom)})")

# Build plant dataset
plant_sequences = [ex["sequence"].upper() for ex in plant_test]
plant_labels = [int(ex["label"]) for ex in plant_test]
print(f"Total plant: {len(plant_sequences)} (positive: {sum(plant_labels)}, negative: {len(plant_labels) - sum(plant_labels)})")

# ===== TOKENIZE BOTH DATASETS =====
from torch.utils.data import TensorDataset, DataLoader

def tokenize_and_make_loader(sequences, labels, name):
    print(f"\nTokenizing {name} ({len(sequences)} sequences, max_length={MAX_LENGTH})...")
    encodings = tokenizer(
        sequences,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    dataset = TensorDataset(
        encodings["input_ids"],
        encodings["attention_mask"],
        torch.tensor(labels, dtype=torch.long),
    )
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    print(f"  Done. Batches: {len(loader)}")
    return loader

mouse_loader = tokenize_and_make_loader(mouse_sequences, mouse_labels, "Mouse")
plant_loader = tokenize_and_make_loader(plant_sequences, plant_labels, "Plant (13 species)")

# ===== EVALUATE ALL MODELS ON BOTH DATASETS =====
def eval_on_dataset(model, loader, model_name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}

models_to_eval = [
    (teacher_model, "Teacher (DNABERT-2)"),
    (hybrid_student, "Hybrid-CNN-3Layer"),
]

# --- Mouse results ---
print(f"\n{'='*80}")
print("CROSS-SPECIES GENERALIZATION: Mouse Promoters (same kingdom: Mammalia)")
print(f"{'='*80}")

mouse_results = {}
for model, name in models_to_eval:
    res = eval_on_dataset(model, mouse_loader, name)
    mouse_results[name] = res
    print(f"  {name:<25} Acc: {res['accuracy']:.2%}  F1: {res['f1']:.4f}  Prec: {res['precision']:.4f}  Rec: {res['recall']:.4f}")

# --- Plant results ---
print(f"\n{'='*80}")
print("CROSS-SPECIES GENERALIZATION: Plant Promoters (different kingdom: Plantae)")
print(f"{'='*80}")

plant_results = {}
for model, name in models_to_eval:
    res = eval_on_dataset(model, plant_loader, name)
    plant_results[name] = res
    print(f"  {name:<25} Acc: {res['accuracy']:.2%}  F1: {res['f1']:.4f}  Prec: {res['precision']:.4f}  Rec: {res['recall']:.4f}")

# ===== FULL COMPARISON TABLE =====
print(f"\n{'='*100}")
print("FULL GENERALIZATION COMPARISON: Human vs Mouse vs Plant")
print(f"{'='*100}")
print(f"{'Model':<25} {'Human Acc':>12} {'Mouse Acc':>12} {'Plant Acc':>12} {'Mouse Drop':>12} {'Plant Drop':>12}")
print("-"*100)

human_accs = {
    "Teacher (DNABERT-2)": teacher_human_acc,
    "Hybrid-CNN-3Layer": hybrid_human_acc,
}

for name in human_accs:
    h = human_accs[name]
    m = mouse_results[name]["accuracy"]
    p = plant_results[name]["accuracy"]
    print(f"{name:<25} {h:>11.2%} {m:>11.2%} {p:>11.2%} {h-m:>11.2%} {h-p:>11.2%}")
print("="*100)
print("\nDrop = Human Acc - Cross-species Acc (lower = better generalization)")

# Speed Benchmark: Inference Latency

In [ ]:
# ===== INFERENCE SPEED BENCHMARK =====
# Measure ms per 1000 sequences for Teacher vs Hybrid Student

from torch.utils.data import DataLoader, Subset

n_bench = min(1000, len(eval_dataset))
bench_subset = Subset(eval_dataset, range(n_bench))

def benchmark_model(model, dataset, batch_size, device_name, num_warmup=3):
    """Benchmark inference speed. Returns ms per 1000 sequences."""
    target_device = torch.device(device_name)
    model = model.to(target_device)
    model.eval()

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # Warmup passes
    for i, batch in enumerate(loader):
        if i >= num_warmup:
            break
        input_ids = batch["input_ids"].to(target_device)
        attention_mask = batch["attention_mask"].to(target_device)
        with torch.no_grad():
            _ = model(input_ids=input_ids, attention_mask=attention_mask)

    if "cuda" in device_name:
        torch.cuda.synchronize()

    # Timed run
    total_samples = 0
    start = time.time()

    for batch in loader:
        input_ids = batch["input_ids"].to(target_device)
        attention_mask = batch["attention_mask"].to(target_device)
        with torch.no_grad():
            _ = model(input_ids=input_ids, attention_mask=attention_mask)
        total_samples += input_ids.shape[0]

    if "cuda" in device_name:
        torch.cuda.synchronize()

    elapsed_ms = (time.time() - start) * 1000
    ms_per_1000 = (elapsed_ms / total_samples) * 1000
    return ms_per_1000, total_samples, elapsed_ms

# ===== BENCHMARK =====
print("="*90)
print("INFERENCE SPEED BENCHMARK: Teacher vs Hybrid Student")
print("="*90)

models_to_bench = [
    ("Teacher (DNABERT-2)", teacher_model),
    ("Hybrid-CNN-3Layer", hybrid_student),
]

gpu_results = {}
cpu_results = {}

# --- GPU (batch_size=16) ---
print(f"\n--- GPU Benchmark (batch_size=16, {n_bench} sequences) ---")
print(f"{'Model':<25} {'Params':>10} {'ms/1000':>12} {'Speedup':>10}")
print("-"*65)

teacher_gpu_ms = None
for name, model in models_to_bench:
    ms_per_1000, total, elapsed = benchmark_model(model, bench_subset, batch_size=16, device_name="cuda")
    gpu_results[name] = ms_per_1000
    if teacher_gpu_ms is None:
        teacher_gpu_ms = ms_per_1000
    speedup = teacher_gpu_ms / ms_per_1000
    params = count_parameters(model)
    print(f"{name:<25} {params/1e6:>8.1f}M {ms_per_1000:>10.1f}ms {speedup:>9.2f}x")

# --- CPU (batch_size=16) ---
print(f"\n--- CPU Benchmark (batch_size=16, {n_bench} sequences) ---")
print(f"{'Model':<25} {'Params':>10} {'ms/1000':>12} {'Speedup':>10}")
print("-"*65)

teacher_cpu_ms = None
for name, model in models_to_bench:
    ms_per_1000, total, elapsed = benchmark_model(model, bench_subset, batch_size=16, device_name="cpu")
    cpu_results[name] = ms_per_1000
    if teacher_cpu_ms is None:
        teacher_cpu_ms = ms_per_1000
    speedup = teacher_cpu_ms / ms_per_1000
    params = count_parameters(model)
    print(f"{name:<25} {params/1e6:>8.1f}M {ms_per_1000:>10.1f}ms {speedup:>9.2f}x")

# Move models back to GPU
for _, model in models_to_bench:
    model.to(device)

# --- Summary ---
print(f"\n{'='*90}")
print("SPEED BENCHMARK SUMMARY")
print(f"{'='*90}")
print(f"{'Model':<25} {'Params':>10} {'GPU ms/1k':>12} {'CPU ms/1k':>12} {'GPU Speedup':>13} {'CPU Speedup':>13}")
print("-"*90)
for name, model in models_to_bench:
    params = count_parameters(model)
    gpu_ms = gpu_results[name]
    cpu_ms = cpu_results[name]
    gpu_sp = teacher_gpu_ms / gpu_ms
    cpu_sp = teacher_cpu_ms / cpu_ms
    print(f"{name:<25} {params/1e6:>8.1f}M {gpu_ms:>10.1f}ms {cpu_ms:>10.1f}ms {gpu_sp:>12.2f}x {cpu_sp:>12.2f}x")
print("="*90)

# Summary & Visualization

In [ ]:
import matplotlib.pyplot as plt

TEACHER_ACCURACY = 0.9671

# ===== CHART 1: Human vs Mouse vs Plant Accuracy =====
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

model_names_short = ["Teacher\n(DNABERT-2)", "Hybrid-CNN\n-3Layer"]
model_keys = ["Teacher (DNABERT-2)", "Hybrid-CNN-3Layer"]
human_acc_vals = [human_accs[k] for k in model_keys]
mouse_acc_vals = [mouse_results[k]["accuracy"] for k in model_keys]
plant_acc_vals = [plant_results[k]["accuracy"] for k in model_keys]

x = np.arange(len(model_names_short))
width = 0.25

bars1 = axes[0].bar(x - width, human_acc_vals, width, label="Human", color="#2ecc71")
bars2 = axes[0].bar(x, mouse_acc_vals, width, label="Mouse", color="#3498db")
bars3 = axes[0].bar(x + width, plant_acc_vals, width, label="Plant (13 spp.)", color="#e74c3c")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Cross-Species Generalization:\nHuman vs Mouse vs Plant Promoters")
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names_short, fontsize=9)
axes[0].legend(fontsize=8)
axes[0].set_ylim(0.4, 1.0)
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{bar.get_height():.1%}", ha="center", fontsize=7)

# ===== CHART 2: GPU Speed =====
gpu_vals = [gpu_results[k] for k in model_keys]
colors = ["gold", "#2ecc71"]
bars4 = axes[1].bar(model_names_short, gpu_vals, color=colors)
axes[1].set_ylabel("ms per 1000 sequences")
axes[1].set_title("GPU Inference Speed\n(lower = faster)")
for bar, ms in zip(bars4, gpu_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, f"{ms:.0f}ms", ha="center", fontsize=9)

# ===== CHART 3: CPU Speed =====
cpu_vals = [cpu_results[k] for k in model_keys]
bars5 = axes[2].bar(model_names_short, cpu_vals, color=colors)
axes[2].set_ylabel("ms per 1000 sequences")
axes[2].set_title("CPU Inference Speed\n(lower = faster)")
for bar, ms in zip(bars5, cpu_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"{ms:.0f}ms", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("/kaggle/working/generalization_speed_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Charts saved to /kaggle/working/generalization_speed_results.png")

# ===== SAVE ALL RESULTS =====
all_results = {
    "human_accuracy": human_accs,
    "mouse_results": mouse_results,
    "plant_results": plant_results,
    "mouse_dataset_info": {
        "source": "DeePromoter (Oubounyt et al., 2019) via CNNPromoterData GitHub",
        "tata_promoters": len(mouse_tata),
        "nontata_promoters": len(mouse_nontata),
        "non_promoters": len(mouse_nonprom),
        "total": len(mouse_sequences),
    },
    "plant_dataset_info": {
        "source": "zhangtaolab/plant-multi-species-core-promoters (HuggingFace)",
        "total_test": len(plant_test),
        "species": dict(species_counts),
    },
    "speed_benchmark": {
        "num_sequences": n_bench,
        "batch_size": 16,
        "gpu": gpu_results,
        "cpu": cpu_results,
        "gpu_speedup": {k: teacher_gpu_ms / gpu_results[k] for k in gpu_results},
        "cpu_speedup": {k: teacher_cpu_ms / cpu_results[k] for k in cpu_results},
    },
    "model_params": {k: count_parameters(m) for k, m in models_to_bench},
}

with open("/kaggle/working/generalization_speed_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print("Results saved to /kaggle/working/generalization_speed_results.json")